In [1]:
import pandas as pd
import json
from api_caller import call_api

In [2]:
df = pd.read_csv("data/EuroParl/preprocessed/full.csv")

In [3]:
def classify_batch(texts):
    joined_text = "\n\n".join(
        [f"ID {i}: {text}" for i, text in enumerate(texts)]
    )

    user_prompt = f"""You are a political ideology detection system.

Your task is to detect STRONG ideological value expression.

A speech should receive a high score ONLY IF it clearly expresses
a political principle or ideological position that could be mapped
onto a political survey (e.g., redistribution, EU integration,
national sovereignty, migration, social equality, market regulation,
democracy, rule of law, etc.).

STRICT CRITERIA:

The speech MUST:
- Advocate or oppose a political principle
- Express how society, the EU, or government SHOULD be structured
- Reveal a stable ideological commitment attributable to the speaker or their party

The speech must NOT be classified as value-expressing if it:
- Expresses generic hope or praise
- Uses polite or diplomatic language
- Evaluates events without ideological reasoning
- Contains general positive or negative sentiment only

IMPORTANT:
Generic approval is NOT ideological.

Be extremely conservative.
If the speech does not clearly state a political principle,
assign a score below 0.2.

Scoring guide:
0.0-0.2 → no ideological value content
0.3-0.5 → weak or vague value signals
0.6-0.8 → clear ideological positioning
0.9-1.0 → strong, explicit ideological commitment

Return STRICT JSON in the same order:
[
  {{
    "score": 0.0-1.0,
    "reason": "..."
  }}
]

IMPORTANT: Return valid JSON array with exactly {len(texts)} entries, 
even if the score is 0.0 for some items.

Texts:
{joined_text}

"""
    response = None
    while True:
      try:
        response = call_api(user_prompt)
        result = json.loads(response["choices"][0]["message"]["content"])
        if len(result) == len(texts):
          return result
        print(f"returned only {len(result)}/{len(texts)}")
      except Exception as e:
        print(f"{e} thrown for {texts}, {response}")
      except:
        print(f"Failed for other reason on {texts}, repeating")


In [4]:
def process_df(df, ifrom, istep=500, batch_size=10):
    j = ifrom
    while j < len(df) - istep:
        try:
            df_subset = df.iloc[j:j+istep].copy()
            indices = list(range(0, len(df_subset), batch_size))
            results = []

            for i in range(0, len(df_subset), batch_size):
                print(f"Progress: {(j+i)/len(df)*100}%")
                batch_texts = df_subset["en"].iloc[i:i+batch_size].tolist()
                batch_results = []
                batch_results = classify_batch(batch_texts)
                results.extend(batch_results)
            
            df_subset["reason"] = [r["reason"] for r in results]
            df_subset["score"] = [r["score"] for r in results]
            final_df = df_subset[["en", "score", "reason", "EU Party"]]
            final_df.to_csv(
                f"data/EuroParl/scored_llama/scored_data_{j}_{j+istep}.csv",
                sep=";",
                index=False,
                encoding="utf-8-sig"
            )

            j += istep
        except Exception as e:
            print(e)

    df_subset = df.iloc[j:].copy()
    indices = list(range(0, len(df_subset), batch_size))
    results = []

    for i in range(0, len(df_subset), batch_size):
        print(f"Progress: {(j+i)/len(df)*100}%")
        batch_texts = df_subset["en"].iloc[i:i+batch_size].tolist()
        batch_results = classify_batch(batch_texts)
        results.extend(batch_results)
    
    df_subset["reason"] = [r["reason"] for r in results]
    df_subset["score"] = [r["score"] for r in results]
    final_df = df_subset[["en", "score", "reason", "EU Party"]]
    final_df.to_csv(
        f"data/EuroParl/scored_llama/scored_data_{j}_{len(df)}.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

In [5]:
process_df(df, ifrom=7000)

Progress: 25.113909518171706%
Progress: 25.1497865317691%
Progress: 25.18566354536648%
Progress: 25.221540558963873%
Progress: 25.25741757256126%
Progress: 25.29329458615865%
Progress: 25.329171599756034%
Progress: 25.365048613353423%
Progress: 25.40092562695081%
Progress: 25.4368026405482%
Progress: 25.47267965414559%
Progress: 25.508556667742976%
Progress: 25.54443368134037%
Progress: 25.58031069493775%
Progress: 25.616187708535143%
Progress: 25.65206472213253%
Progress: 25.68794173572992%
Progress: 25.723818749327304%
Progress: 25.759695762924693%
Expecting value: line 1 column 1 (char 0) thrown for ['By adopting this directive, the European Union is allowing internet services to be cut off without the need for a judicial order.', 'I welcome the fact that this text will increase users’ rights to universal services, via clearer contracts, a more accessible emergency telephone number, a hotline for missing children, greater consideration of the rights of disabled people, and a guarant